# 🧘 AntarJyoti User Feedback Database Viewer
This notebook provides simple SQL commands to query and analyze the user feedback telemetry stored inside the SQLite database (`chroma.sqlite3`).

In [ ]:
import os
import sqlite3
import pandas as pd

# 1. Setup path to local SQLite database
# Tries to find Gemini DB first, then Ollama DB
db_path = 'backend/chroma_db_backup/chroma.sqlite3'
if not os.path.exists(db_path):
    db_path = 'backend/chroma_db_ollama/chroma.sqlite3'

if os.path.exists(db_path):
    print(f"✅ Successfully located database at: {db_path}")
else:
    print(f"❌ Could not find database file at {db_path}. Please make sure you are running this from the repository root.")

## 📊 1. Fetch All Feedback Data
Runs a simple `SELECT *` and loads it into a Pandas DataFrame for easy viewing.

In [ ]:
conn = sqlite3.connect(db_path)

# Load feedback table to Pandas DataFrame
query = "SELECT id, question, feedback_value, latency_ms, timestamp FROM user_feedback ORDER BY timestamp DESC;"
df = pd.read_sql_query(query, conn)
conn.close()

# Render the table
df

## 📈 2. Aggregate Feedback & Latency Statistics
Summarizes helpfulness scores and response times.

In [ ]:
conn = sqlite3.connect(db_path)

print("--- Telemetry Summary ---")
stats_df = pd.read_sql_query("""
    SELECT 
        COUNT(*) as total_responses,
        SUM(case when feedback_value = 1 then 1 else 0 end) as helpful_count,
        SUM(case when feedback_value = -1 then 1 else 0 end) as unhelpful_count,
        ROUND(AVG(latency_ms) / 1000.0, 2) as avg_latency_seconds,
        ROUND(MIN(latency_ms) / 1000.0, 2) as min_latency_seconds,
        ROUND(MAX(latency_ms) / 1000.0, 2) as max_latency_seconds
    FROM user_feedback;
""", conn)

conn.close()
stats_df

## 🔍 3. Custom SQL Query Sandbox
You can modify the query below to inspect the full entries, answers, and sources.

In [ ]:
conn = sqlite3.connect(db_path)

# Example: Fetch only negative feedback to see what needs improvement
custom_query = """
    SELECT question, answer, latency_ms 
    FROM user_feedback 
    WHERE feedback_value = -1;
"""

df_custom = pd.read_sql_query(custom_query, conn)
conn.close()
df_custom